# 13-2절 연습 문제 풀이

이 노트북은 13-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch13/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 13장 공통 - CLIP + GPT-2 이미지 설명 생성기
# 주의: 모델과 데이터셋 내려받기가 필요하다.
try:
    from transformers import (CLIPModel, CLIPProcessor, GPT2LMHeadModel,
                              GPT2Tokenizer, Blip2Processor,
                              Blip2ForConditionalGeneration)
    from datasets import load_dataset
except ImportError:
    print('알림: pip install transformers datasets pillow 가 필요하다.')

CLIP_NAME = 'openai/clip-vit-base-patch32'
GPT2_NAME = 'gpt2'
BLIP2_NAME = 'Salesforce/blip2-opt-2.7b'

## 연습 13-7

BLIP-2의 시각 질의응답 기능을 사용해 한 장의 이미지를 두고 최소 다섯 가지 서로 다른 종류의 질문(예: 사물 식별, 색깔, 개수, 위치, 행동, 상황 해석 등)을 만들어 답을 생성해 보자. 어떤 질문에는 잘 답하고 어떤 질문에는 잘 답하지 못하는지 정리해 보자.

In [ ]:
processor = Blip2Processor.from_pretrained(BLIP2_NAME)
model = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_NAME, torch_dtype=torch.float16, device_map='auto')

from PIL import Image
image = Image.open('../../data/cat.jpg').convert('RGB')
QUESTIONS = [
    ('사물 식별', 'Question: What animal is in the image? Answer:'),
    ('색깔',     'Question: What color is the animal? Answer:'),
    ('개수',     'Question: How many animals are there? Answer:'),
    ('위치',     'Question: Where is the animal? Answer:'),
    ('행동',     'Question: What is the animal doing? Answer:'),
    ('상황 해석', 'Question: What is the mood of this photo? Answer:'),
]
for kind, q in QUESTIONS:
    inputs = processor(images=image, text=q, return_tensors='pt').to(
        model.device, torch.float16)
    out = model.generate(**inputs, max_new_tokens=20)
    print(f'[{kind}] {processor.decode(out[0], skip_special_tokens=True).strip()}')

BLIP-2는 **사물 식별·색깔·행동**처럼 이미지에서 직접 확인할 수 있는 질문에 강하다. 반면 **개수 세기**는 자주 틀리고(3개 이상이면 특히), **상황 해석·감정 추론**처럼 배경지식이 필요한 질문은 뻔한 답을 내놓는 경향이 있다.

개수 세기가 약한 이유는 Q-Former의 쿼리 토큰 32개가 이미지를 요약하는 과정에서 **정확한 개수 정보가 보존되지 않기** 때문이다.

## 연습 13-8

BLIP-2의 한국어 처리 성능을 확인해 보자. 'What is in the image?'를 한국어 '이미지에 무엇이 있나요?'로 바꿔 같은 이미지의 답을 생성해 보고, 영어 답변과 비교해 보자. 결과의 차이가 발생하는 이유를 'BLIP-2의 구현 방식과 Q-Former' 본문 설명에 비추어 추론해 보자.

힌트: 본문에서 BLIP-2가 사용하는 언어 모델과 관련된 설명을 다시 확인해 보자.

In [ ]:
for lang, q in [('영어', 'Question: What is in the image? Answer:'),
                ('한국어', '질문: 이미지에 무엇이 있나요? 답변:')]:
    inputs = processor(images=image, text=q, return_tensors='pt').to(
        model.device, torch.float16)
    out = model.generate(**inputs, max_new_tokens=30)
    print(f'[{lang}] {processor.decode(out[0], skip_special_tokens=True).strip()}')

한국어 질문에는 답변 품질이 크게 떨어지거나 영어로 답한다.

**이유**: BLIP-2의 언어 모델인 **OPT-2.7B가 영어 중심으로 사전 학습**되었기 때문이다. Q-Former는 이미지를 잘 요약하지만, 그 결과를 언어로 풀어내는 것은 언어 모델의 몫이다. 즉 멀티모달 모델의 언어 능력은 **결합된 언어 모델의 능력을 넘지 못한다**.

한국어가 필요하면 한국어 LLM을 언어 모델로 쓰는 구성([연습 문제 13-5])이 필요하다.

## 연습 13-9

[도전 문제] BLIP-2와 함께 자주 비교되는 모델로 LLaVALarge Language and Vision Assistant가 있다. 허깅페이스 모델 허브에서 LLaVA 모델 카드를 찾아 BLIP-2와 어떻게 다른지(특히 비전 인코더와 언어 모델 사이를 잇는 모듈의 구조 측면에서) 정리해 보자.

13장 학습 노트

### 풀이

| 구분 | BLIP-2 | LLaVA |
|---|---|---|
| 연결 방식 | **Q-Former**(학습 가능한 쿼리 32개 + 크로스 어텐션) | **단순 MLP**(선형 계층 2개) |
| 비전 인코더 | ViT-g/14 (EVA-CLIP) | CLIP ViT-L/14 |
| 언어 모델 | OPT, Flan-T5 등 | Vicuna, Llama 계열 |
| 학습 단계 | 2단계(표현 학습 → 생성 학습) | 1~2단계(사전 정렬 → 시각 지시 튜닝) |
| 전달 토큰 수 | 32개(고정) | 패치 수만큼(수백 개) |

**핵심 차이**

- **BLIP-2**는 이미지 정보를 32개 토큰으로 **압축**한다. 언어 모델에 들어가는 토큰이 적어 효율적이지만, 세부 정보(개수, 작은 글자)가 손실될 수 있다.
- **LLaVA**는 패치 임베딩을 거의 그대로 언어 모델에 넘긴다. 정보 손실이 적어 세밀한 이해에 강하지만 시퀀스가 길어져 연산이 무겁다.
- **LLaVA의 진짜 강점은 시각 지시 튜닝(visual instruction tuning)** 이다. GPT-4로 생성한 대화형 데이터로 학습해 자유로운 대화와 추론에 능하다.

정리하면 BLIP-2는 **연결 구조를 정교하게**, LLaVA는 **연결은 단순하게 두고 데이터와 언어 모델에 투자**하는 방향이다. 최근에는 후자가 더 좋은 결과를 내고 있다.